# LFCC-BiLSTM

Linear Frequency Cepstral Coefficients (LFCC) fed into a Bidirectional LSTM.
LFCC uses a **linearly-spaced** filterbank instead of the mel scale, making it
more sensitive to high-frequency detail.

## 1. Imports & Config

In [1]:
import os
import numpy as np
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm


In [2]:
ROOT_PATH = "/kaggle/input/datasets/artharking/severity-dau-kdah/Severity_DAU_KDAH"  # change this

SR = 16000
N_LFCC = 20          # number of LFCC coefficients
FRAME_LEN = 0.025     # 25 ms
HOP_LEN   = 0.010     # 10 ms

FIXED_LEN  = 398    # time frames

# BiLSTM hyper-params
HIDDEN_SIZE = 256
NUM_LAYERS  = 2
DROPOUT     = 0.3

BATCH_SIZE = 32
EPOCHS     = 20
LR         = 1e-3


In [3]:
label_map = {
    "N": 0,
    "L": 1,
    "M": 2,
    "H": 3
}
NUM_CLASSES = len(label_map)


In [4]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


## 2. Feature Extraction — LFCC

Pipeline: STFT → triangular linear filterbank → log energy → DCT

In [5]:
from scipy.fftpack import dct

def extract_lfcc(file_path):
    """
    Linear Frequency Cepstral Coefficients (LFCC).
    Uses a linearly-spaced filterbank (unlike MFCC's mel scale).
    Steps: STFT → linear filterbank → log energy → DCT → LFCC
    """
    y, sr = librosa.load(file_path, sr=SR)
    n_fft      = int(FRAME_LEN * sr)
    hop_length = int(HOP_LEN  * sr)

    # 1. Power spectrogram
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)) ** 2
    # S: (n_fft//2+1, T)

    # 2. Linear filterbank  (N_LFCC filters, evenly spaced in Hz)
    n_freqs = S.shape[0]
    filters = np.zeros((N_LFCC, n_freqs))
    freq_bins = np.linspace(0, sr // 2, n_freqs)
    centers   = np.linspace(0, sr // 2, N_LFCC + 2)
    for k in range(N_LFCC):
        lo, mid, hi = centers[k], centers[k+1], centers[k+2]
        filters[k] = np.maximum(0,
            np.minimum((freq_bins - lo)/(mid - lo + 1e-10),
                        (hi - freq_bins)/(hi - mid + 1e-10)))

    # 3. Log filterbank energy
    fb_energy = np.dot(filters, S)          # (N_LFCC, T)
    log_fb    = np.log(fb_energy + 1e-10)

    # 4. DCT → cepstral coefficients
    lfcc = dct(log_fb, type=2, axis=0, norm='ortho')[:N_LFCC, :]

    # 5. Fix length
    if lfcc.shape[1] < FIXED_LEN:
        lfcc = np.pad(lfcc, ((0,0),(0,FIXED_LEN-lfcc.shape[1])), mode='constant')
    else:
        lfcc = lfcc[:, :FIXED_LEN]
    return lfcc   # (N_LFCC, FIXED_LEN)


## 3. Precompute & Save Features

In [6]:
SAVE_PATH = "/kaggle/working/lfcc_features"
os.makedirs(SAVE_PATH, exist_ok=True)

def save_lfcc_dataset(root_dir, split):
    split_path = os.path.join(root_dir, split)
    for severity in os.listdir(split_path):
        sev_path = os.path.join(split_path, severity)
        if not os.path.isdir(sev_path): continue
        save_sev = os.path.join(SAVE_PATH, split, severity)
        os.makedirs(save_sev, exist_ok=True)
        for file in tqdm(os.listdir(sev_path), desc=f'{split}-{severity}'):
            if not file.endswith('.wav'): continue
            np.save(os.path.join(save_sev, file.replace('.wav','.npy')),
                    extract_lfcc(os.path.join(sev_path, file)))

save_lfcc_dataset(ROOT_PATH, 'train')
save_lfcc_dataset(ROOT_PATH, 'test')


test-L: 100%|██████████| 79/79 [00:01<00:00, 63.51it/s]


## 4. Dataset & DataLoaders

In [7]:
import torch
from torch.utils.data import Dataset

class LfccDataset(Dataset):
    def __init__(self, root_dir):
        self.files  = []
        self.labels = []
        for severity in os.listdir(root_dir):
            sev_path = os.path.join(root_dir, severity)
            if not os.path.isdir(sev_path): continue
            for file in os.listdir(sev_path):
                if file.endswith('.npy'):
                    self.files.append(os.path.join(sev_path, file))
                    self.labels.append(label_map[severity])

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        lfcc = np.load(self.files[idx])          # (N_FEAT, FIXED_LEN)
        # BiLSTM expects (seq_len, input_size) → transpose to (FIXED_LEN, N_FEAT)
        lfcc = torch.tensor(lfcc.T, dtype=torch.float32)  # (FIXED_LEN, N_FEAT)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return lfcc, label


In [8]:
train_dataset = LfccDataset("/kaggle/working/lfcc_features/train")
test_dataset  = LfccDataset("/kaggle/working/lfcc_features/test")

val_size   = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_ds, val_ds = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 5. BiLSTM Model

In [9]:
import torch
import torch.nn as nn

drop_amount = 0.255

class BiLSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=drop_amount if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(drop_amount)

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):

        # Initial hidden and cell states
        h0 = torch.zeros(
            self.num_layers * 2,
            x.size(0),
            self.hidden_size,
            device=x.device
        )

        c0 = torch.zeros(
            self.num_layers * 2,
            x.size(0),
            self.hidden_size,
            device=x.device
        )

        # LSTM output
        out, _ = self.lstm(x, (h0, c0))

        out = self.dropout(out)

        # Forward last timestep + backward first timestep
        last_hidden_state = torch.cat(
            (
                out[:, -1, :self.hidden_size],
                out[:, 0, self.hidden_size:]
            ),
            dim=1
        )

        output = self.fc(last_hidden_state)

        return output

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTMClassifier(
    input_size=N_LFCC,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES
).to(device)


## 6. Training

In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

import shutil

best_val_acc = 0
save_path = "/tmp/best_model.pth"

for epoch in range(EPOCHS):

    # ================= TRAIN =================
    model.train()
    train_loss = 0

    for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [train]'):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    # ================= VALIDATION =================
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in val_loader:

            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()

            total += y.size(0)

    val_acc = correct / total

    print(f'Epoch {epoch+1}/{EPOCHS}: Loss={train_loss:.4f}  Val Acc={val_acc:.4f}')

    # ================= SAVE BEST =================
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            save_path,
            _use_new_zipfile_serialization=False
        )

        print(f'✅ Best model saved (Val Acc={val_acc:.4f})')

shutil.copy(save_path, './best_model.pth')

print('✅ Model copied to working directory!')

Model params: 2,148,356


Epoch 1/20 [train]: 100%|██████████| 289/289 [00:22<00:00, 12.77it/s]


Epoch 1/20: Loss=190.3491  Val Acc=0.8322
✅ Best model saved (Val Acc=0.8322)


Epoch 2/20 [train]: 100%|██████████| 289/289 [00:22<00:00, 12.60it/s]


Epoch 2/20: Loss=153.0139  Val Acc=0.8546
✅ Best model saved (Val Acc=0.8546)


...

Epoch 18/20: Loss=24.6732  Val Acc=0.9766


Epoch 19/20 [train]: 100%|██████████| 289/289 [00:23<00:00, 12.35it/s]


Epoch 19/20: Loss=16.7560  Val Acc=0.9795
✅ Best model saved (Val Acc=0.9795)


Epoch 20/20 [train]: 100%|██████████| 289/289 [00:23<00:00, 12.35it/s]


Epoch 20/20: Loss=16.3452  Val Acc=0.9805
✅ Best model saved (Val Acc=0.9805)
✅ Model copied to working directory!


## 7. Evaluation

In [12]:
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

correct = total = 0
all_preds  = []
all_labels = []

with torch.no_grad():
    for x, y in tqdm(test_loader, desc='Test'):
        x, y  = x.to(device), y.to(device)
        preds = torch.argmax(model(x), dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

print(f'✅ Test Accuracy: {correct/total:.4f}')


Test: 100%|██████████| 84/84 [00:02<00:00, 31.66it/s]

✅ Test Accuracy: 0.9168


In [13]:
from sklearn.metrics import classification_report
print('\nClassification Report:\n')
print(classification_report(all_labels, all_preds, target_names=list(label_map.keys())))



Classification Report:

              precision    recall  f1-score   support

           N       0.99      0.99      0.99      2037
           L       0.18      0.28      0.22        79
           M       0.81      0.82      0.81       506
           H       0.00      0.00      0.00        59

    accuracy                           0.92      2681
   macro avg       0.49      0.52      0.51      2681
weighted avg       0.91      0.92      0.91      2681



In [14]:
# import pandas as pd

# inv_map = {v: k for k, v in label_map.items()}

# filenames = [os.path.basename(test_dataset.files[i]) for i in range(len(test_dataset))]

# scores = []
# all_preds = []
# all_labels = []

# model.eval()
# with torch.no_grad():
#     for x, y in tqdm(DataLoader(test_dataset, batch_size=32, shuffle=False)):
#         x, y = x.to(device), y.to(device)
#         outputs = model(x)
#         probs = torch.softmax(outputs, dim=1)
#         best_scores = probs.max(dim=1).values
#         preds = torch.argmax(outputs, dim=1)

#         scores.extend(best_scores.cpu().numpy())
#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(y.cpu().numpy())

# df = pd.DataFrame({
#     "filename":      filenames,
#     "score":         scores,
#     "predict class": [inv_map[p] for p in all_preds],
#     "actual class":  [inv_map[l] for l in all_labels],
# })

# df.to_excel("/kaggle/working/results.xlsx", index=False)
# print("✅ Saved results.xlsx")

import pandas as pd

inv_map = {v: k for k, v in label_map.items()}

# Class order: Normal → High → Mid → Low → Very_Low
class_order = ["N", "L", "M", "H"]

filenames  = [os.path.basename(test_dataset.files[i]) for i in range(len(test_dataset))]

all_preds      = []
all_labels     = []
all_raw_scores = []   # raw logits per class
all_softmax    = []   # softmax probabilities per class

model.eval()
with torch.no_grad():
    for x, y in tqdm(DataLoader(test_dataset, batch_size=32, shuffle=False)):
        x, y = x.to(device), y.to(device)
        outputs = model(x)                          # raw logits: (batch, NUM_CLASSES)
        probs   = torch.softmax(outputs, dim=1)     # softmax probabilities
        preds   = torch.argmax(outputs, dim=1)

        all_raw_scores.extend(outputs.cpu().numpy())
        all_softmax.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# Build DataFrame
df = pd.DataFrame({"filename": filenames})

# Raw logit score for each class
for cls_name in class_order:
    df[f"score_{cls_name}"] = [row[label_map[cls_name]] for row in all_raw_scores]

# Softmax probability for each class
for cls_name in class_order:
    df[f"prob_{cls_name}"] = [row[label_map[cls_name]] for row in all_softmax]

# Final prediction & actual label
df["predict class"] = [inv_map[p] for p in all_preds]
df["actual class"]  = [inv_map[l] for l in all_labels]

# Save both CSV and Excel
df.to_csv("/kaggle/working/results.csv",   index=False)
df.to_excel("/kaggle/working/results.xlsx", index=False)
print("✅ Saved results.csv and results.xlsx")
print(df.head())


100%|██████████| 84/84 [00:02<00:00, 30.49it/s]


✅ Saved results.csv and results.xlsx
               filename   score_N   score_L   score_M   score_H    prob_N  \
0  M22_Hindi_H_0019.npy -3.055554  0.194858  3.593446 -1.659403  0.001246   
1  M22_Hindi_H_0049.npy -2.589379  3.500574  0.988375 -2.832198  0.002088   
2  M22_Hindi_H_0047.npy -3.062693  0.349133  3.863028 -2.706098  0.000952   
3  M22_Hindi_H_0042.npy -4.157553  1.893703  3.594273 -2.981888  0.000363   
4  M22_Hindi_H_0017.npy -3.897028  0.778943  2.176795  1.347990  0.001366   

     prob_L    prob_M    prob_H predict class actual class  
0  0.032137  0.961586  0.005032             M            H  
1  0.921547  0.074728  0.001638             L            H  
2  0.028853  0.968836  0.001359             M            H  
3  0.154153  0.844307  0.001176             M            H  
4  0.146576  0.593120  0.258939             M            H  


## 8. Single-File Prediction

In [15]:
# def predict(file_path, model_path='best_model.pth'):
#     m = BiLSTMModel(N_LFCC, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT)
#     m.load_state_dict(torch.load(model_path, map_location='cpu'))
#     m.eval()
#     feat = torch.tensor(extract_lfcc(file_path).T, dtype=torch.float32).unsqueeze(0)
#     with torch.no_grad():
#         pred = torch.argmax(m(feat), dim=1).item()
#     return {v:k for k,v in label_map.items()}[pred]
